In [22]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import json

path = "/content/drive/MyDrive/dataset/sirene_full.json"
with open(path, "r", encoding="utf-8") as f:
    entreprises = json.load(f)

print("Nombre d'entreprises :", len(entreprises))
print(entreprises[0])

Nombre d'entreprises : 50
{'siren': '310658265', 'nic': '00031', 'siret': '31065826500031', 'dateCreationEtablissement': '2005-04-02', 'etablissementSiege': True, 'etatAdministratifUniteLegale': 'C', 'dateCreationUniteLegale': '1900-01-01', 'denominationUniteLegale': None, 'categorieJuridiqueUniteLegale': '1000', 'activitePrincipaleUniteLegale': '71.11Z', 'nomenclatureActivitePrincipaleUniteLegale': 'NAFRev2', 'categorieEntreprise': None, 'enseigneEtablissement': None, 'sexeUniteLegale': 'M', 'nomUniteLegale': 'MAISONABE', 'nomUsageUniteLegale': None, 'prenom1UniteLegale': 'JEAN-LOUIS', 'prenomUsuelUniteLegale': 'JEAN-LOUIS', 'adresseEtablissement': 'RUE D’ATHENES, 12000 RODEZ FRANCE', 'complementAdresseEtablissement': 'BOURRAN VILLA HERMES', 'numeroVoieEtablissement': None, 'indiceRepetitionEtablissement': None, 'typeVoieEtablissement': 'RUE', 'libelleVoieEtablissement': 'D’ATHENES', 'codePostalEtablissement': '12000', 'libelleCommuneEtablissement': 'RODEZ', 'metadata': {'pipeline_pro

In [25]:
import random
from faker import Faker

fake = Faker("fr_FR")

BANKS = [
    {"name": "BNP Paribas", "code_banque": "30004", "bic": "BNPAFRPP", "logo": "bnp.png"},
    {"name": "Société Générale", "code_banque": "30003", "bic": "SOGEFRPP", "logo": "sg.png"},
    {"name": "Crédit Agricole", "code_banque": "30006", "bic": "AGRIFRPP", "logo": "ca.png"},
    {"name": "La Banque Postale", "code_banque": "20041", "bic": "PSSTFRPP", "logo": "lbp.png"}
]

def compute_rib_key(bank, branch, account):
    def convert(c):
        if c.isdigit():
            return c
        return str(ord(c.upper()) - 55)

    account_num = "".join(convert(c) for c in account)
    rib_number = f"{bank}{branch}{account_num}"
    key = 97 - (int(rib_number) % 97)
    return f"{key:02d}"

def compute_iban(bank, branch, account, rib_key):
    def convert(c):
        if c.isdigit():
            return c
        return str(ord(c.upper()) - 55)

    account_num = "".join(convert(c) for c in account)

    rib = f"{bank}{branch}{account_num}{rib_key}"
    temp = rib + "1527"
    iban_key = 98 - (int(temp) % 97)
    return f"FR{iban_key:02d}{rib}"

def get_rib_holder(company):
    denom = company.get("denominationUniteLegale")

    if denom and denom.strip():
        return denom.strip(), "entreprise"

    nom = company.get("nomUniteLegale")
    prenom = company.get("prenom1UniteLegale") or company.get("prenomUsuelUniteLegale")

    if nom and prenom:
        return f"{prenom.strip()} {nom.strip()}", "personne"

    if nom:
        return nom.strip(), "personne"

    return "Titulaire Inconnu", "inconnu"

def get_address(company):
    return company.get("adresseEtablissement") or (
        f"{company.get('numeroVoieEtablissement','')}"+
        f"{company.get('typeVoieEtablissement','')}"+
        f"{company.get('libelleVoieEtablissement','')}, "+
        f"{company.get('codePostalEtablissement','')}"+
        f"{company.get('libelleCommuneEtablissement','')}"
    )

def generate_rib(company):
    bank = random.choice(BANKS)

    code_banque = bank["code_banque"]
    code_guichet = str(random.randint(10000, 99999))
    account = "".join(random.choices("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=11))

    rib_key = compute_rib_key(code_banque, code_guichet, account)
    iban = compute_iban(code_banque, code_guichet, account, rib_key)

    holder, type_holder = get_rib_holder(company)

    return {
        "titulaire": holder,
        "type": type_holder,
        "adresse": get_address(company),

        "banque": bank["name"],
        "logo_banque": bank["logo"],
        "bic": bank["bic"],

        "code_banque": code_banque,
        "code_guichet": code_guichet,
        "numero_compte": account,
        "cle_rib": rib_key,
        "iban": iban,

        "agence": fake.city()
    }

In [26]:
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def rib_to_pdf(rib, filename):
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()

    elements = []

    elements.append(Paragraph("<b>RELEVÉ D'IDENTITÉ BANCAIRE</b>", styles["Title"]))
    elements.append(Spacer(1, 12))

    elements.append(Paragraph(f"<b>Titulaire :</b> {rib['titulaire']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Adresse :</b> {rib['adresse']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>Banque :</b> {rib['banque']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Agence :</b> {rib['agence']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>BIC :</b> {rib['bic']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>Code Banque :</b> {rib['code_banque']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Code Guichet :</b> {rib['code_guichet']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Numéro de compte :</b> {rib['numero_compte']}", styles["Normal"]))
    elements.append(Paragraph(f"<b>Clé RIB :</b> {rib['cle_rib']}", styles["Normal"]))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph(f"<b>IBAN :</b> {rib['iban']}", styles["Normal"]))

    doc.build(elements)

In [27]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/dataset/rib_pdf")
output_dir.mkdir(parents=True, exist_ok=True)

for i, company in enumerate(entreprises):
    rib = generate_rib(company)

    siret = company.get("siret", f"{i}")
    pdf_path = output_dir / f"rib_{siret}.pdf"

    rib_to_pdf(rib, str(pdf_path))
